# NegotiateEnv Training - Full Episodes - OpenEnv Hackathon

**Team**: Mayuka Reddy & Kushal Adhyaru  
**Environment**: https://kushaladhyaru-negotiate-env.hf.space  
**Repository**: https://github.com/MadhaviSG/openEnv-negotiateEnv

---

## Configuration

- **Baseline episodes**: 200 (all scenarios)
- **Training episodes**: 500 (2.5x coverage of 200 scenarios)
- **Method**: Unsloth 4-bit LoRA with GRPO
- **Actual baseline**: 0.4833 (from 200-episode test)
- **Training target**: 0.55-0.60 (15-25% improvement)

## 1. Check GPU

In [ ]:
# Check GPU type and memory
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 2. Load Repository

**Option A**: Upload from your local machine  
**Option B**: Clone from GitHub

In [ ]:
# Option A: Upload from local machine
# 1. Click the folder icon in the left sidebar
# 2. Click the upload button
# 3. Upload your entire openEnv-negotiateEnv folder as a zip file
# 4. Then run this:

import os

# If you uploaded a zip file
if os.path.exists('openEnv-negotiateEnv.zip'):
    print("Extracting uploaded repository...")
    !unzip -q openEnv-negotiateEnv.zip
    %cd openEnv-negotiateEnv
    print("Repository loaded from local upload!")
# If you uploaded the folder directly
elif os.path.exists('openEnv-negotiateEnv'):
    print("Using uploaded repository...")
    %cd openEnv-negotiateEnv
    print("Repository loaded from local upload!")
# Otherwise clone from GitHub
else:
    print("No local repository found. Cloning from GitHub...")
    !git clone https://github.com/MadhaviSG/openEnv-negotiateEnv.git
    %cd openEnv-negotiateEnv
    print("Repository cloned from GitHub!")

# Verify files
!ls -la train_negotiate_unsloth.py evaluate.py

## 3. Install Dependencies

In [ ]:
# Install the negotiate_env package
!pip install -q -e .

# Install basic dependencies
!pip install -q requests matplotlib numpy

print("\nBasic dependencies installed!")

## 4. Test Connection to Environment

In [ ]:
# Your deployed environment URL
ENV_URL = "https://kushaladhyaru-negotiate-env.hf.space"

print(f"Environment URL: {ENV_URL}")
print("Testing connection...\n")
print("="*60)

import requests

# Test reset endpoint (main endpoint we need)
try:
    response = requests.post(f"{ENV_URL}/reset", json={}, timeout=10)
    if response.status_code == 200:
        print("[OK] Environment server is accessible!")
        obs = response.json()
        print(f"   Scenario: {obs.get('context', '')[:80]}...")
    else:
        print(f"[ERROR] Server returned status code: {response.status_code}")
        print(f"   Response: {response.text[:200]}")
except Exception as e:
    print(f"[ERROR] Failed to connect: {e}")
    print("\nMake sure your Space is running!")
    print(f"Visit: {ENV_URL}")

print("\n" + "="*60)

## 5. Run Full Baseline Evaluation (200 Episodes)

This will evaluate the rule-based agent on ALL 200 scenarios from your dataset.

In [ ]:
# Full baseline evaluation on 200 episodes
print("Running FULL baseline evaluation (200 episodes)...")
print("This covers all scenarios in your dataset.\n")
print("Expected time: ~5-10 minutes\n")
print("="*60)

!python evaluate.py --agent rule --episodes 200 --env-url {ENV_URL}

print("\n" + "="*60)
print("\nFull baseline evaluation complete!")
print("\nThis is your benchmark to beat with training.")

## 6. Install Training Dependencies

In [ ]:
# Remove conflicting packages
print("Cleaning up conflicting packages...\n")
!pip uninstall -y -q vllm 2>/dev/null || true

# Install Unsloth
print("Installing Unsloth...\n")
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install TRL and dependencies (use latest stable version)
print("Installing TRL and dependencies...\n")
!pip install -q "trl>=0.11.0" transformers accelerate peft datasets

print("\n" + "="*60)
print("All dependencies installed!")
print("="*60)

## 7. Run Training (500 Episodes)

**Training Configuration:**
- Model: Qwen/Qwen2.5-1.5B-Instruct (4-bit quantized)
- Episodes: 500 (covers all 200 scenarios 2.5 times)
- Max turns per episode: 10 (same as baseline)
- Time on H100: ~30-40 minutes
- Time on T4: ~3-4 hours
- Expected improvement: 15-25% over baseline

**Your baseline from 200 episodes: 0.4833**  
**Training target: 0.55-0.60**

In [ ]:
# Run training with 500 episodes
print("Starting Unsloth 4-bit LoRA training with GRPO...")
print("Training on 500 episodes (2.5x coverage of 200 scenarios)\n")
print("Expected time:")
print("  - H100: ~30-40 minutes")
print("  - T4: ~3-4 hours\n")
print("You can monitor progress below.")
print("The notebook will continue automatically when done.\n")
print("="*60)

!python train_negotiate_unsloth.py \
    --env-url {ENV_URL} \
    --model-id Qwen/Qwen2.5-1.5B-Instruct \
    --output-dir negotiate-unsloth-output \
    --num-episodes 500 \
    --max-turns 10

print("\n" + "="*60)
print("Training complete!")
print("Model saved to: negotiate-unsloth-output/")
print("="*60)

## 8. Plot Training Results

In [ ]:
# Plot reward curve
import os
from IPython.display import Image, display

print("Generating reward curve...\n")

if os.path.exists("negotiate-unsloth-output/trainer_state.json"):
    !python plot_reward_curve.py \
        --log-file negotiate-unsloth-output/trainer_state.json \
        --out reward_curve.png
    
    if os.path.exists("reward_curve.png"):
        print("Reward curve generated!\n")
        display(Image("reward_curve.png"))
    else:
        print("[WARN] Reward curve image not found")
else:
    print("[WARN] Training log not found. Make sure training completed successfully.")

## 9. Training Summary & Comparison

In [ ]:
# Display comprehensive training summary
import json
import os

print("="*60)
print("TRAINING SUMMARY & COMPARISON")
print("="*60)

trainer_state_path = "negotiate-unsloth-output/trainer_state.json"

if os.path.exists(trainer_state_path):
    with open(trainer_state_path) as f:
        state = json.load(f)
    
    log_history = state.get("log_history", [])
    
    # Extract rewards
    rewards = []
    for entry in log_history:
        reward = entry.get("env_reward") or entry.get("reward") or entry.get("train/env_reward") or entry.get("train/reward")
        if reward is not None:
            rewards.append(float(reward))
    
    if rewards:
        print(f"\nTraining Method: Unsloth 4-bit LoRA with PPO")
        print(f"Total Episodes: {len(rewards)}")
        print(f"\n--- Training Progress ---")
        print(f"Initial Reward: {rewards[0]:.4f}")
        print(f"Final Reward: {rewards[-1]:.4f}")
        print(f"Best Reward: {max(rewards):.4f}")
        print(f"Average Reward: {sum(rewards)/len(rewards):.4f}")
        
        improvement = rewards[-1] - rewards[0]
        improvement_pct = (improvement / rewards[0] * 100) if rewards[0] > 0 else 0
        print(f"\nImprovement: {improvement:.4f} ({improvement_pct:.1f}%)")
        
        # Compare with baseline (from your 200-episode test)
        baseline = 0.4833  # Your actual baseline
        print(f"\n--- Comparison with Baseline ---")
        print(f"Baseline (Rule-based, 200 episodes): {baseline:.4f}")
        print(f"Trained Model (Final): {rewards[-1]:.4f}")
        
        if rewards[-1] > baseline:
            beat_baseline = rewards[-1] - baseline
            beat_pct = (beat_baseline / baseline * 100)
            print(f"\n[SUCCESS] Beat baseline by: {beat_baseline:.4f} ({beat_pct:.1f}%)")
            print(f"\nTrained model outperforms baseline!")
        else:
            gap = baseline - rewards[-1]
            gap_pct = (gap / baseline * 100)
            print(f"\n[WARN] Below baseline by: {gap:.4f} ({gap_pct:.1f}%)")
            print("\nPossible reasons:")
            print("- Training needs more episodes (try 1000+)")
            print("- Learning rate too high/low")
            print("- Model needs more training time")
        
        # Show progress milestones
        print(f"\n--- Training Milestones ---")
        milestones = [0, len(rewards)//4, len(rewards)//2, 3*len(rewards)//4, len(rewards)-1]
        for i, idx in enumerate(milestones):
            if idx < len(rewards):
                print(f"Episode {idx:3d}: {rewards[idx]:.4f}")
    else:
        print("\n[WARN] No reward data found in training logs")
else:
    print("\n[WARN] Training state not found")
    print("   Make sure training completed successfully")

print("\n" + "="*60)

## 10. Run Demo

In [ ]:
# Run demo negotiation
print("Running demo negotiation...\n")
print("="*60)

!python demo.py --difficulty medium --env-url {ENV_URL}

print("\n" + "="*60)

## 11. Save Model to HuggingFace Hub

In [ ]:
# Install huggingface_hub
!pip install -q huggingface_hub

In [ ]:
# Login to HuggingFace using Colab secrets
from huggingface_hub import login
from google.colab import userdata

print("Logging in to HuggingFace...\n")
print("Make sure you've added 'Huggingface_Token' to Colab secrets:")
print("1. Click the key icon in the left sidebar")
print("2. Add a new secret named 'Huggingface_Token'")
print("3. Paste your token from: https://huggingface.co/settings/tokens\n")

try:
    hf_token = userdata.get('Huggingface_Token')
    login(token=hf_token)
    print("Successfully logged in to HuggingFace!")
except Exception as e:
    print(f"[ERROR] Failed to get Huggingface_Token from secrets: {e}")
    print("\nFalling back to interactive login...\n")
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# Push model to HuggingFace Hub
from huggingface_hub import HfApi
import os

api = HfApi()

output_dir = "negotiate-unsloth-output"
repo_id = "KushalAdhyaru/negotiate-env-qwen-unsloth-500ep"

if os.path.exists(output_dir):
    print(f"Uploading {output_dir} to {repo_id}...\n")
    print("This may take a few minutes...\n")
    print("="*60)
    
    try:
        api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_id,
            repo_type="model",
        )
        print("\n" + "="*60)
        print("Model uploaded successfully!")
        print(f"\nView your model at: https://huggingface.co/{repo_id}")
        print("="*60)
    except Exception as e:
        print("\n" + "="*60)
        print(f"[ERROR] Upload failed: {e}")
        print("\nYou can manually upload later.")
        print("="*60)
else:
    print(f"[ERROR] Output directory not found: {output_dir}")

## 12. Download Results

In [ ]:
# Create results archive
print("Creating results archive...\n")

!zip -r training_results_500ep.zip \
    reward_curve.png \
    negotiate-unsloth-output/ \
    *.log 2>/dev/null || true

print("\nResults saved to training_results_500ep.zip")
print("\nDownload from the file browser (left sidebar)")

## Final Summary

### What You Accomplished:

1. Full baseline evaluation (200 episodes on all scenarios)
2. Comprehensive training (500 episodes, 2.5x dataset coverage)
3. Reward curve visualization
4. Model saved to HuggingFace Hub
5. Complete performance comparison

### Your Results:

- **Baseline**: 0.4833 (rule-based, 200 episodes)
- **Trained**: Check summary above
- **Improvement**: Check comparison above

### Hackathon Submission:

- Environment: https://kushaladhyaru-negotiate-env.hf.space
- Dataset: https://huggingface.co/datasets/mayukareddy/SyntheticSaasDataset
- Model: https://huggingface.co/KushalAdhyaru/negotiate-env-qwen-unsloth-500ep
- Code: https://github.com/MadhaviSG/openEnv-negotiateEnv
- Training results and visualizations

---

**Good luck with your hackathon submission!**